In [1]:
using Pkg
Pkg.activate(".")
include("Tools.jl")
include("KrylovTechnical.jl")
include("GaugeFixing.jl");
include("./Lab/newton-step-SR.jl");

  Activating project at `~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R`
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """


In [2]:
gilt_eps = 1e-4 #from the paper for this chi
chi = 10
cg_eps = 1e-10
gilt_pars = Dict(
	"gilt_eps" => gilt_eps,
	"cg_chis" => collect(1:chi),
	"cg_eps" => cg_eps,
	"verbosity" => 0,
	"rotate" => true,
)
Jratio = 1.0

relT=1.0
#do 3 steps from the critical tensor
initialA_pars = Dict("relT" => relT, "Jratio" => Jratio)
traj = trajectory(initialA_pars, 3, gilt_pars)["A"];
#NB traj consists of PyObjects

traj = traj .|> x -> fix_continuous_gauge(x)[1]; #this is still PyObjects
traj[4], accepted_elements, _ = fix_discrete_gauge(traj[4]; tol = 1e-7);

function fix_discrete_by_accepted_elements_if_possible(x)
	res = x
	try
		res = fix_discrete_gauge(x, accepted_elements)[1]
	catch
		res = fix_discrete_gauge(x)[1]
	end
	return res
end

traj = traj .|> x -> fix_discrete_by_accepted_elements_if_possible(x);
traj = py_to_ju.(traj);
traj = traj .|> x -> x / norm(x); 

/Users/slava/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GiltTNR/GiltTNR2D_Ising_benchmarks.py:169: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return sinh(2*x*Jv)*sinh(2*x*Jh) - 1
/Users/slava/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GiltTNR/GiltTNR2D_Ising_benchmarks.py:169: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return sinh(2*x*Jv)*sinh(2*x*Jh) - 1
/Users/slava/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GiltTNR/GiltTNR2D_Ising_benchmarks.py:169: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array 

Newton iterations below (I interrupted the code after a few iterations, but in previous runs I saw it converge)

In [ ]:
A = Any[ NaN for _ in 1:40 ]; # list of tensors, Newton method trajectory
accepted_elements = Any[ NaN for _ in 1:40 ]; # list of elements in gauge-fixing
deltaA = Any[ NaN for _ in 1:40 ]; # list of deltaA's proposed by Newton method

A[1] = traj[4]
for i in 1:20
    A[i], accepted_elements[i] = fix_discrete_gauge(A[i]; tol = 1e-7);
    RA = gilt(A[i], accepted_elements[i], gilt_pars);
    println("i=",i) 
    println("||R(A[i])-A[i]||= ", embedded_distance(RA, A[i]))
    flush(stdout)
    #deltaA[i] = newton_correction(A[i], 5, accepted_elements[i], gilt_pars);
    deltaA[i] = newton_correction_with_iterations_fixed(A[i], 5, accepted_elements[i], gilt_pars);
    println("||deltaA[i]||= ", norm(deltaA[i]))
    A[i+1] = A[i] + deltaA[i]
end

Below tests showing that Jacobian spectrum does not vary from one run to the other (if using newton_correction, spectrum varies a bit)

In [5]:
deltaA1 = newton_correction_with_iterations_fixed(A[1], 5, accepted_elements[1], gilt_pars);

Dict{Any, Any}((1, "N") => 38, (1, "W") => 24, (1, "S") => 41, (1, "E") => 25, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


┌ Info: Arnoldi eigsolve finished after 3 iterations:
│ *  11 eigenvalues converged
│ *  norm of residuals = (1.0238839928130892e-46, 2.1129013801130613e-31, 5.08381410239299e-31, 1.6963710259467642e-19, 1.718177285873873e-16, 5.583264004460139e-16, 5.583264004460139e-16, 2.6275561559824944e-16, 2.6275561559824944e-16, 4.743902047772559e-16, 4.743902047772559e-16)
└ *  number of operations = 43


EIGENVALUES (INITIAL):
1.9972981648015007 + 0.0im
-0.9162210914779763 + 0.0im
-0.9131300461950596 + 0.0im
0.44216944217651105 + 0.0im
-0.3345516741636342 + 0.0im
-4.498347677370572e-5 + 0.3120485917884362im
-4.498347677370572e-5 - 0.3120485917884362im
-0.07263141717757504 + 0.29393337230076366im
-0.07263141717757504 - 0.29393337230076366im
0.0726290568881181 + 0.29382949491370497im
0.0726290568881181 - 0.29382949491370497im


In [7]:
deltaA1 = newton_correction_with_iterations_fixed(A[1], 5, accepted_elements[1], gilt_pars);

Dict{Any, Any}((1, "N") => 38, (1, "W") => 24, (1, "S") => 41, (1, "E") => 25, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


┌ Info: Arnoldi eigsolve finished after 2 iterations:
│ *  5 eigenvalues converged
│ *  norm of residuals = (4.82616535345107e-35, 1.448881848709182e-22, 1.163534383695808e-22, 1.2727329238726033e-15, 1.3511589807924597e-13)
└ *  number of operations = 34


EIGENVALUES (INITIAL):
1.9972992604920927 + 0.0im
-0.916221084667822 + 0.0im
-0.9131264524875634 + 0.0im
0.4421691350097396 + 0.0im
-0.33455939505121624 + 0.0im
